# Plot MCWF squeezing analysis

Load one globally analyzed trajectory ensemble and plot its group-resolved moments and squeezing.

In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "common").is_dir() and (candidate / "slurm").is_dir():
            return candidate
    raise RuntimeError("Could not find repo root containing common/ and slurm/.")


repo_root = find_repo_root(Path.cwd().resolve())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from common.plotting import plot_bloch_angles, plot_spin_components, plot_squeezing
from slurm.j_moments_io import load_j_moments_artifact

In [ ]:
omega_1_label = "1.0"
filename = f"v1_omega_{omega_1_label}"
input_dir = repo_root / "slurm" / "analyses" / "squeezing_omega1-scan" / "output"

In [ ]:
artifact = load_j_moments_artifact(input_dir / f"{filename}.pkl")
moments = artifact["moments"]

if moments.metadata is None or moments.J is None or moments.squeezing is None:
    raise ValueError("The artifact does not contain complete squeezing moments.")

phase_protocol = moments.metadata.phase_protocol
group_labels = tuple(
    f"Group {group_index + 1}"
    for group_index in range(len(moments.squeezing))
)

In [ ]:
plot_bloch_angles(
    moments.t,
    moments.J.theta,
    moments.J.phi,
    labels=group_labels,
    phase_protocol=phase_protocol,
    title=rf"$J$-sphere angles, $\omega_1={omega_1_label}$",
)

plot_spin_components(
    moments.t,
    moments.J.x,
    moments.J.y,
    moments.J.z,
    moments.J.length,
    labels=group_labels,
    phase_protocol=phase_protocol,
    title=rf"$J$-sphere components, $\omega_1={omega_1_label}$",
)

plot_squeezing(
    moments.t,
    tuple(series.xi2 for series in moments.squeezing),
    tuple(series.lambda_min for series in moments.squeezing),
    tuple(series.N_c for series in moments.squeezing),
    tuple(series.excited_fraction_active for series in moments.squeezing),
    labels=group_labels,
    phase_protocol=phase_protocol,
    title=rf"Generalized squeezing, $\omega_1={omega_1_label}$",
)